# Exposure map of DAMPE

In [ ]:
import numpy as np
import os
from matplotlib import pyplot as plt

plt.rcParams['figure.dpi'] = 400
plt.rcParams['savefig.dpi'] = 300

dirDir = "/home/users/l/lebrun/"
dataPath = dirDir + "data/"
figPath = dirDir + "figures/"
resultPath = dirDir + "results/"
print(dataPath)

## Load data from Jennifer

In [ ]:
filename = "Exposure_DAMPE_8years_tot.npy"
data = np.load(dataPath + filename)


### Inspect data

In [ ]:
print(data.shape)
print(data[0:4,:])


for i in range(data.shape[1]) :
    print(f"Column {i} spans from {np.min(data[:,i])} to {np.max(data[:,i])}")

# Extract columns
energy = data[:, 0]
longitudes = data[:, 3]
latitudes = data[:, 4]

print(f"Shape of lons : {longitudes.shape} ; shape of lats : {latitudes.shape}")
# 0 is an intensity, 1 is RA, 2 is DEC, 3 is l, 4 is b, 5 is theta

longitudes_converted = -(longitudes - 2*np.pi*(longitudes//np.pi))
print(longitudes_converted)
print(latitudes)

In [ ]:

# Compute bin edges
lon_min, lon_max = longitudes_converted.min(), longitudes_converted.max()
lat_min, lat_max = latitudes.min(), latitudes.max()

res_n = 1.5

# Define the edges of the bins for longitude and latitude
lon_bins = np.linspace(-np.pi, np.pi, int(360*res_n))  # Adjust the number of bins as needed
lat_bins = np.linspace(-np.pi/2, np.pi/2, int(180*res_n))

# Create 2D histogram (aggregates energy values)
energy_grid, lon_edges, lat_edges = np.histogram2d(
    longitudes_converted, latitudes, 
    bins=[lon_bins, lat_bins], 
    weights=energy
)

# Compute bin centers for plotting
lon_centers = (lon_edges[:-1] + lon_edges[1:]) / 2
lat_centers = (lat_edges[:-1] + lat_edges[1:]) / 2




### Plot Map

In [ ]:

fig = plt.figure(figsize=(8, 5))
ax = fig.add_subplot(111, projection="mollweide")
ax.set_facecolor('black')


# Create pcolormesh plot
mesh = ax.pcolormesh(lon_centers, lat_centers, energy_grid.T, 
                        cmap='viridis',  # You can change colormap
                        shading='auto')

# Add colorbar
plt.colorbar(mesh, ax=ax, label='Energy')

ax.xaxis.set_tick_params(labelcolor="white")
tick_labels = np.array([150, 120, 90, 60, 30, 0, -30, -60, -90, -120, -150])
ax.set_xticklabels(tick_labels)
ax.set_xlabel(r"$l$ [deg]")
ax.set_ylabel(r"$b$ [deg]")
ax.grid(True, linestyle="--", alpha=0.5)
fig.savefig(figPath+"exposureMap.pdf")

## Normalize map

In [ ]:
max_E = np.max(energy_grid)
print(max_E)
energy_grid_normalized = energy_grid / max_E


### Save grid

In [ ]:
print(energy_grid_normalized.shape)
print(lon_edges.shape)
print(lat_edges.shape)

lon_mesh, lat_mesh = np.meshgrid(lon_centers, lat_centers)

# Reshape everything to 1D arrays
energies = energy_grid_normalized.T.flatten()  # Transpose to match meshgrid orientation
lons = lon_mesh.flatten()
lats = lat_mesh.flatten()

# Stack into a single array
result_array = np.column_stack((energies, lons, lats))# Save to file
np.savetxt(resultPath + f'normalized_exposure_map_n={res_n}.csv', result_array, delimiter=',', header='energy,longitude,latitude')
np.save(resultPath + f'normalized_exposure_map_n={res_n}.npy', energy_grid_normalized)

### Plot Normalized Map

In [ ]:
fig2 = plt.figure(figsize=(8, 5))
ax2 = fig2.add_subplot(111, projection="mollweide")
ax2.set_facecolor('black')


# Create pcolormesh plot
mesh_normalized = ax2.pcolormesh(lon_centers, lat_centers, energy_grid_normalized.T, 
                        cmap='viridis',  # You can change colormap
                        shading='auto')

# Add colorbar
plt.colorbar(mesh_normalized, ax=ax2, label='Energy')

ax2.xaxis.set_tick_params(labelcolor="white")
tick_labels = np.array([150, 120, 90, 60, 30, 0, -30, -60, -90, -120, -150])
ax2.set_xticklabels(tick_labels)
ax2.set_xlabel(r"$l$ [deg]")
ax2.set_ylabel(r"$b$ [deg]")
ax2.grid(True, linestyle="--", alpha=0.5)
fig2.savefig(figPath+"exposureMapNormalized.pdf")